In [5]:
"""
story_to_animation_ltx.py
 
Music-video animation pipeline.
 
Stage 1 — Whisper transcribes the input song (sliding window, ffmpeg+soundfile,
           no librosa/numba dependency).
Stage 2 — OpenAI turns the lyrics into scene_description + action_sequence entries,
           grouped at verse/paragraph level (not per sentence).
Stage 3 — Z-Image (Tongyi-MAI/Z-Image-Turbo) renders one 1280×704 still per scene.
Stage 4 — LTX-2.3 A2VidPipelineTwoStage animates each still with (image + text +
           audio slice) → video.  No new audio is synthesised; the original song
           slice is muxed directly into each clip.  Two seeded variations are
           produced per scene for your manual pick step.
 
Key improvements over the previous version (sourced from the working
story_to_animation.py implementation):
 
  • Pipeline is instantiated via the PYTHON API (constructor + __call__), NOT the
    argparse CLI.  The CLI's --images / --distilled-lora argv encodings have
    already changed between repo revisions and keep breaking.  The Python API is
    documented and stable.
 
  • QuantizationPolicy is imported from ltx_core.quantization via its
    classmethod (QuantizationPolicy.fp8_cast()), which does NOT mmap the 46 GB
    checkpoint — avoiding "Cannot allocate memory" failures on strict-overcommit
    systems.  Falls back gracefully to older import paths.
 
  • Pre-quantized fp8 checkpoints (filename contains "fp8") are detected and
    loaded with quantization=None — applying fp8-cast on top of an already-fp8
    file causes a KeyError on the gate layers' missing input_scale tensors.
 
  • OffloadMode is imported from ltx_pipelines.utils.types with a try/except
    fallback, instead of being assumed present.
 
  • The distilled checkpoint sets DISTILLED_SIGMAS as stage_1_sigmas and uses
    guidance-free MultiModalGuiderParams (cfg_scale=1.0, stg_scale=0.0) — the
    dev checkpoint path uses STG (cfg_scale=3.0, stg_scale=1.0, stg_blocks=[29]).
 
  • call_supported() introspects the pipeline's __call__ signature and passes
    only the kwargs it actually accepts, so minor upstream signature drift
    degrades gracefully instead of crashing.
 
  • torch.no_grad() (not torch.inference_mode()) wraps generation because the
    spatial upsampler's conv3d path raises "Inference tensors cannot be saved for
    backward" under inference_mode on some revisions.
 
  • Asset auto-discovery (find_ltx2_assets) scans the repo's models/ directory
    and HuggingFace cache so you don't have to hard-code every path.
 
  • LoraPathStrengthAndSDOps is only constructed when the NON-distilled (dev)
    checkpoint is used — the distilled checkpoint does not need it.
 
  • ImageConditioningInput is preferred over a plain tuple for the images arg;
    falls back to (path, 0, 1.0) for older revisions.
 
Run inside the LTX-2 repo environment (~/repos/LTX-2) or with ltx-core and
ltx-pipelines pip-installed into the active conda/system environment:
    pip install -e ~/repos/LTX-2/packages/ltx-core
    pip install -e ~/repos/LTX-2/packages/ltx-pipelines
"""
 
import gc
import inspect
import json
import json_repair
import numpy as np
import os
import random
import re
import secrets
import soundfile as sf
import subprocess
import tempfile
import time
import torch
import traceback
 
from datetime import datetime
from difflib import SequenceMatcher
from openai import OpenAI
from PIL import Image
from diffusers import ZImagePipeline
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from moviepy.audio.io.AudioFileClip import AudioFileClip
 
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["XFORMERS_IGNORE_FLASH_VERSION_CHECK"] = "1"

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype  = torch.bfloat16
 
MAX_SEED    = np.iinfo(np.int32).max
retry_limit = 3
 
THEME = "cinematic with CGI special effects, 3D animation, intimate dratamtic emotional scene, fourth of july, patiotic, red white and blue"
 

In [6]:
# =============================================================================
# CONFIGURATION
# =============================================================================
LTX_ROOT   = os.path.expanduser("~/repos/LTX-2")
LTX_MODELS = os.path.join(LTX_ROOT, "models")
 
# "4090" → Ada Lovelace, native FP8, 128 GB RAM
# "3090" → Ampere, no native FP8, 64 GB RAM
MACHINE = "4090"
 
_MACHINE_PRESETS = {
    "4090": {
        # bf16 distilled checkpoint + fp8-cast: the correct combo for cpu offload.
        # fp8-scaled-mm is NOT compatible with block-streaming / cpu offload.
        # If you have the pre-quantized fp8 file (ltx-2.3-22b-distilled-fp8.safetensors)
        # the asset discovery below will find it; the code then sets quantization=None
        # automatically (double-quantising a pre-quantized file breaks gate layers).
        "ltx_quantization": "fp8-cast",
        "ltx_offload":      "cpu",
    },
    "3090": {
        # Ampere has no FP8 tensor cores — fp8-cast still works (weight-only cast).
        # 64 GB RAM is tight with cpu offload; switch to "disk" if OOM.
        "ltx_quantization": "fp8-cast",
        "ltx_offload":      "cpu",
    },
}

CONFIG = {
    "openai_api_key": "",
    "openai_model": "gpt-5.4-nano",
    "openai_model_small_reasoning": "gpt-5.4-nano",
    "openai_model_large": "gpt-5.4-nano",
    "hf_token": "",
    "base_working_dir": "./images",
    "base_video_dir": "./output",
    "audio_files": [
        "/mnt/d/Share/Audio/BombsBursting.mp3",    
    ],
    "device": device,
    "dtype": dtype,
    "retry_limit": retry_limit,
    "MAX_SEED": MAX_SEED,
    # ---- LTX-2.3 model paths (auto-discovered if left blank) ----------------
    # Leave blank to use find_ltx2_assets() auto-discovery, or set explicitly.
    "ltx_checkpoint":              "",   # e.g. "ltx-2.3-22b-distilled-1.1.safetensors"
    "ltx_spatial_upsampler":       "",   # e.g. "ltx-2.3-spatial-upscaler-x2-1.1.safetensors"
    "ltx_distilled_lora":          "",   # only needed for the DEV (non-distilled) checkpoint
    "ltx_distilled_lora_strength": 0.8,
    "ltx_gemma_root":              "",   # e.g. "models/gemma-3-12b"
 
    # ---- LTX-2.3 generation settings ----
    "ltx_width": 1280,          # two-stage requires multiples of 64 (stage 1 runs at half res).
    "ltx_height": 704,          # 1280x704 -> stage1 640x352, both /32 -> valid.
    "ltx_fps": 24,
    "ltx_num_inference_steps": 8,   # stage-1 steps; lower = faster.
    "ltx_negative_prompt": (
        "shaky, glitchy, low quality, worst quality, deformed, distorted, disfigured, "
        "motion smear, motion artifacts, fused fingers, bad anatomy, weird hand, ugly, "
        "transition, static, watermark, text, letters, subtitles, titles, words, scene change"
    ),
    "ltx_enhance_prompt": False,
 
    # ---- workflow ----
    "ltx_num_variations": 1,    # you make >=2 animations per scene and pick manually.
    "ltx_max_scene_seconds": 12.0,  # matches your existing 12s scene splitter.
    "ltx_min_scene_seconds": 3.0,
}

CONFIG["ltx_checkpoint"]   = os.path.join(LTX_MODELS, "ltx-2.3/ltx-2.3-22b-distilled-1.1.safetensors")
#CONFIG["ltx_quantization"] = "fp8-cast"   # casts bf16 to fp8 on the fly, no mmap
CONFIG["ltx_quantization"] = "none"
CONFIG["ltx_offload"]      = "cpu"        # block streaming: only active block in VRAM
CONFIG["ltx_a2v_guidance_scale"] = 1.0
CONFIG["ltx_enhance_prompt"]     = False

# try EROS
CONFIG["ltx_checkpoint"] = os.path.join(
    LTX_MODELS, "ltx-2.3/10Eros_v1.4_bf16.safetensors"
)
CONFIG["ltx_distilled_lora"] = os.path.join(
    LTX_MODELS, "ltx-2.3/ltx-2.3-22b-distilled-lora-1.1_fro90_ceil72_condsafe.safetensors"
)
CONFIG["ltx_distilled_lora_strength"] = 1.0  # condsafe is safe at full strength for I2V
CONFIG["ltx_quantization"] = "fp8-cast"
CONFIG["ltx_offload"] = "cpu"

# Fold the machine preset in.
CONFIG.update(_MACHINE_PRESETS[MACHINE])
 
# Z-Image still resolution matches the LTX target so the conditioning frame
# is not cropped or distorted.
HEIGHT = CONFIG["ltx_height"]
WIDTH  = CONFIG["ltx_width"]
 
outputs_folder = "./temp_outputs/"
os.makedirs(outputs_folder,                  exist_ok=True)
os.makedirs(CONFIG["base_working_dir"],      exist_ok=True)
os.makedirs(CONFIG["base_video_dir"],        exist_ok=True)
 
# =============================================================================
# Prompt example libraries  — paste your full blocks here unchanged.
# =============================================================================
SCENE_DESCRIPTIONS = '''
scene_description: Soft digital anime-style illustration of a young person with short blonde hair standing in a golden wheat field at sunset
Subject: A youth with shoulder-length soft pinkish-blonde bobbed hair, large expressive eyes looking downward gently toward the guitar strings
Clothing: Wearing an open teal hooded windbreaker over a white graphic t-shirt and rolled-up dark blue denim shorts; carrying a black backpack on one shoulder
Action: Standing in profile holding an acoustic wooden guitar with both hands poised to play near the fretboard, head slightly bowed in quiet concentration or emotion
Environment: Surrounded by tall golden wheat stalks swaying gently under warm ambient light; distant hazy mountains line the horizon beneath a pastel sky filled with fluffy pink and lavender clouds dusted with sparkling star-like particles
Lighting: Golden-hour backlight from low sun creating soft lens flare, casting gentle highlights on hair and jacket while leaving subtle shadows for depth; overall mood is dreamy and romanticized through glowing particle effects floating around the figure.

scene_description: Oil painting of a silhouetted figure standing on rocky ground facing a large white sun with swirling red energy ring overhead.
Subject: Silhouetted figure with back to viewer.
Clothing: Dark tattered clothing with glowing red markings.
Objects: Katana held low at side.
Environment: Traditional Japanese structure in silhouette against misty mountain range background.
Lighting: Soft atmospheric with high contrast between bright sky and shadowed foreground.
Style Details: Loose expressive brushstrokes enhancing epic dramatic fantasy mood.

scene_description: Comic book art; a mysterious woman in profile peeking from behind a brick wall while holding a revolver. 
Subject: A stylish female spy with sharp blue eyes and red lipstick, wearing a wide-brimmed purple fedora that casts shadows across her face. She has dark wavy hair falling over one shoulder. 
Clothing: Purple trench coat buttoned at the collar with visible belt buckle; matching purple gloves gripping the gun handle tightly. The fabric appears smooth with subtle creases indicating movement. 
Action: Her body is angled sideways as she peeks around a corner, eyes focused forward while holding her weapon ready but not yet firing suggesting stealth and tension rather than immediate combat. 
Environment: Urban alleyway setting featuring weathered purple-gray bricks on the right side of frame; dark vertical drainpipe runs along left edge suggesting narrow passage between buildings. Background wall shows peeling paint texture adding gritty atmosphere typical of noir cityscapes. 
Lighting: Strong directional light from front-left creates dramatic contrast illuminating her cheekbone, nose bridge and hand while leaving other areas in deep shadow to enhance mystery and focus attention on facial features. Purple color palette dominates throughout scene enhancing moody nighttime ambiance without explicit darkness indicators like stars or moonlight sources being shown directly within composition itself though overall impression strongly implies low-light conditions typical of urban nightscape scenarios depicted through stylized rendering techniques common in graphic novels featuring female protagonists operating covertly during evening hours when surveillance systems might be active yet insufficient to detect hidden observers moving silently among shadows cast by architectural elements surrounding them at this particular moment captured here for dramatic effect within narrative context implied solely through visual storytelling methods employed consistently across entire artwork without relying upon dialogue boxes or textual exposition present elsewhere in source material from which original image was derived initially before undergoing post-processing adjustments made specifically tailored towards achieving desired aesthetic outcomes achieved successfully herein today. 
Style Details: Bold black outlines define all forms clearly separating foreground figure sharply against darker background elements; flat color application minimizes gradients except where necessary for shading purposes around hat brim and coat folds creating sense of volume without excessive detail cluttering up clean lines characteristic of classic comic book illustration style reminiscent of mid-twentieth century superhero comics adapted here into modern thriller genre conventions blending traditional-artistic techniques with contemporary digital tools used widely nowadays by independent creators worldwide producing high-quality works independently outside mainstream publishing industry constraints typically associated with corporate-owned franchises dominating global markets today.

scene_description: A digital illustration of a wide-eyed purple-striped Cheshire Cat with neon yellow eyes and an exaggerated grin floating in the center
Subject: a striped cat character resembling Alice's Cheshire Cat, sitting upright amidst swirling colors
Environment: a surreal vortex tunnel made of flowing liquid paint textures blending deep purples, magentas, blues, and bright cyans that curve inward toward a glowing white light source at the back
Objects: floating playing cards (aces of spades, hearts, diamonds) spinning through the air along with large red-and-white spotted mushrooms resting on invisible ground planes within the swirl
Lighting: dramatic rim lighting from behind creating strong outlines and highlights while casting deep shadows for high contrast against a dark background filled with glowing particles. 
Style Details: hyper-detailed digital painting style reminiscent of fantasy art or animated movie posters using vibrant saturated colors.

scene_description: Subject: A lone figure of Batman viewed from behind, wearing a dark cowl and textured cape with jagged edges
Environment: Standing in the middle of a flooded, ruined cityscape at night under a stormy sky filled with bats flying overhead
Lighting: Dimly lit by glowing red embers rising from burning ruins and scattered floating sparks reflecting on wet ground surfaces;

scene_description: A macro digital illustration of a tiny bioluminescent lizard perched on the tip of an index finger against a blurred city street at night. 
Subject: A small gecko with glowing blue and pink skin, large expressive golden eyes, and delicate whiskers; its body is covered in subtle sparkles and faint orange spots along the neck and limbs. 
Action: The gecko sits calmly on a human fingertip, gazing upward toward the viewer or slightly off-frame with curiosity; its tiny claws grip gently but firmly around the skin ridge where it rests. 
Lighting: Vibrant bioluminescent glow emanates from within the creature's body, casting soft pink and blue reflections onto surrounding surfaces; background lights create bokeh highlights that enhance depth and mood without overpowering the subject. 
Environment: A vibrant urban nightscape with streaks of neon signs in cyan, magenta, yellow, and red forming a colorful backdrop; distant buildings are softly blurred to emphasize foreground focus while suggesting motion or traffic below. 
Camera: Close-up framing capturing fine details like skin texture and eye reflection; shallow depth of field isolates the gecko from its bustling surroundings for intimate storytelling impact. 
Style Details: Highly stylized digital art with smooth gradients, glowing particle effects around edges, and a dreamy bokeh aesthetic blending fantasy elements into realistic textures for an otherworldly yet grounded feel.

scene_description: Close-up hyper-realistic portrait of a rugged warrior with long white hair and beard in a snowy environment
Subject: Weathered male face with deep-set amber eyes, prominent forehead wrinkles, thick dark eyebrows dusted with frosty snowflakes, sharp nose, full lips slightly parted. 
Clothing: Dark textured clothing partially visible at the neck, blending into shadows. 
Action: Staring intensely forward, slight tension in facial muscles suggesting determination or weariness from battle. 
Environment: Falling snowflakes around face and hair creating depth against a dark blurred background hinting at cold winter landscape. 
Lighting: Soft directional lighting highlighting facial features with subtle highlights on forehead and cheekbones contrasting deeper shadows under jawline. 
Style Details: Cinematic composition, high detail rendering of skin texture and frost effects, dramatic atmosphere emphasizing resilience in harsh conditions.
'''
 
ACTION_SEQUENCES = (
    "action_sequence: The man dances energetically, leaping mid-air with fluid arm swings and quick footwork\n"
    "action_sequence: The girl skateboarding, repeating the endless spinning and dancing and jumping on a skateboard, with clear movements, full of charm\n"
    "action_sequence: The man dances flamboyantly, swinging his hips and striking bold poses with dramatic flair\n"
    "action_sequence: The young man writes intensely, flipping papers and adjusting his glasses with swift, focused movements\n"
    "action_sequence: The girl dances gracefully, with clear movements, full of charm\n"
    "action_sequence: A jellyfish dances in the sea\n"
    "action_sequence: A pretty clown girl with blue skin giggles as she twirls a knife between her fingers.\n"
)

In [7]:

# =============================================================================
# Utility / memory
# =============================================================================
def reset_memory(dev):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(dev)
        torch.cuda.reset_accumulated_memory_stats(dev)
        torch.cuda.ipc_collect()
 
 
def safe_json_parse(response_text):
    """Robustly extract and parse a JSON list from an LLM response."""
    cleaned_text = response_text.strip()
    if cleaned_text.startswith("```"):
        lines = cleaned_text.splitlines()
        if len(lines) >= 2:
            if lines[0].startswith("```"):
                lines = lines[1:]
            if lines[-1].startswith("```"):
                lines = lines[:-1]
            cleaned_text = "\n".join(lines)
    try:
        data = json_repair.loads(cleaned_text)
        if isinstance(data, list):
            # Flatten one level if the LLM wrapped scenes in an extra list
            if data and isinstance(data[0], list):
                data = [item for sublist in data for item in sublist]
            data = [d for d in data if isinstance(d, dict)]
            if data:
                return data
        elif isinstance(data, dict):
            for key in data:
                if isinstance(data[key], list):
                    inner = data[key]
                    if inner and isinstance(inner[0], list):
                        inner = [item for sublist in inner for item in sublist]
                    inner = [d for d in inner if isinstance(d, dict)]
                    if inner:
                        return inner
            return [data]
    except Exception as e:
        print(f"JSON Repair failed: {e}")
    try:
        match = re.search(r'\[.*\]', response_text, re.DOTALL)
        if match:
            data = json_repair.loads(match.group())
            if isinstance(data, list):
                if data and isinstance(data[0], list):
                    data = [item for sublist in data for item in sublist]
                data = [d for d in data if isinstance(d, dict)]
                return data
    except Exception:
        pass
    return []
 
def _truncate_prompt(prompt: str, max_words: int = 300) -> str:
    """Truncate prompt to avoid Gemma 1024 token limit."""
    words = prompt.split()
    if len(words) > max_words:
        truncated = " ".join(words[:max_words])
        print(f"  [prompt] Truncated from {len(words)} to {max_words} words.")
        return truncated
    return prompt
     
# =============================================================================
# Audio helpers — ffmpeg + soundfile (no librosa, no numba)
# =============================================================================
def load_audio_mono_16k(audio_file: str) -> tuple:
    """Decode any audio format to mono float32 at 16 kHz via ffmpeg+soundfile."""
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-i", audio_file,
             "-ar", "16000", "-ac", "1", "-f", "wav", tmp_path],
            check=True, capture_output=True,
        )
        audio, sr = sf.read(tmp_path, dtype="float32", always_2d=False)
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)
    return audio, sr
 
 
def slice_audio(full_audio_path: str, start_sec: float, duration_sec: float,
                out_path: str) -> str:
    """Cut a time slice from the song and write it as WAV at original quality."""
    audio, sr = sf.read(full_audio_path, dtype="float32", always_2d=True)
    audio = audio.T   # (samples, ch) -> (ch, samples)
    n_ch, n_samp = audio.shape
    start  = int(start_sec * sr)
    end    = min(int((start_sec + duration_sec) * sr), n_samp)
    chunk  = audio[:, start:end]
    needed = int(round(duration_sec * sr))
    if chunk.shape[1] < needed:
        pad   = np.zeros((n_ch, needed - chunk.shape[1]), dtype=chunk.dtype)
        chunk = np.concatenate([chunk, pad], axis=1)
    sf.write(out_path, chunk.T, sr)
    return out_path
 
 
def get_audio_duration(audio_file: str) -> float:
    with AudioFileClip(audio_file) as clip:
        return clip.duration
 
 
# =============================================================================
# Whisper transcription
# =============================================================================
def get_audio_pipe():
    model_id  = "openai/whisper-large-v3-turbo"
    processor = WhisperProcessor.from_pretrained(model_id)
    model     = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=dtype, low_cpu_mem_usage=True
    ).to(device)
    return processor, model
 
 
# =============================================================================
# Z-Image still generation
# =============================================================================
def load_zimage_models():
    pipe = ZImagePipeline.from_pretrained(
        "Tongyi-MAI/Z-Image-Turbo",
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=False,
    )
    pipe.to("cuda")
    return pipe
 
 
@torch.inference_mode()
def generate_image(pipe: ZImagePipeline, prompt: str, negative_prompt: str,
                   resolution: tuple, seed: int):
    prompt = THEME + ", " + prompt
    height, width = resolution
    if seed == -1:
        seed = secrets.randbits(32)
    image = pipe(
        prompt=prompt,
        height=height,
        width=width,
        num_inference_steps=9,
        guidance_scale=0.0,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]
    return image, seed
 
 
# =============================================================================
# OpenAI helpers
# =============================================================================
def get_openai_prompt_response(prompt: str, config: dict, max_tokens: int = 12000,
                               temperature: float = 0.33, openai_model: str = ""):
    client   = OpenAI(api_key=config["openai_api_key"])
    response = client.chat.completions.create(
        max_completion_tokens=max_tokens,
        messages=[
            {"role": "system", "content": "Act as a helpful assistant, you are an expert editor."},
            {"role": "user",   "content": prompt},
        ],
        model=openai_model or config["openai_model"],
    )
    retry_count = 0
    while retry_count < config["retry_limit"]:
        try:
            return response.choices[0].message.content
        except Exception as e:
            print(f"Error occurred: {e}")
            retry_count += 1
            if retry_count == config["retry_limit"]:
                print("Retry limit reached.")
                return ""
            print(f"Retrying... ({retry_count}/{config['retry_limit']})")
            time.sleep(1)
 
 
def generate_character_profile(text: str, config: dict):
    print("Generating Character Profile...")
    prompt = (
        f"Analyze these lyrics and create a VISUAL CHARACTER PROFILE for the main hero.\n"
        f"Lyrics: {text[:300]}...\n"
        "Output ONLY a 50-word description of their Age, Gender, Clothing, and Vibe."
    )
    try:
        profile = get_openai_prompt_response(
            prompt, config, openai_model=config["openai_model"], temperature=0.7
        )
        return profile.strip().replace('"', '')
    except Exception:
        return "A resilient hero in a distressed olive hoodie and dark jeans."
 
 
def similarity_ratio(a, b):
    return SequenceMatcher(None, a, b).ratio()
 
 
def create_scenes(text: str, character_profile: str, config: dict):
    prompt = (
        "### ROLE\n"
        "You are a visionary director of photography creating a screenplay for a short film. "
        "Translate song lyrics into a JSON list of visually stunning, diverse scenes.\n\n"
        "### INPUT DATA\n"
        f"- **Theme:** {THEME}\n"
        f"- **Lyrics:** {text}\n"
        f"- **Main Character Profile:** \"{character_profile}\"\n\n"
        "### VISUAL STYLE & GUIDELINES\n"
        "1. Mix scenes WITH and WITHOUT the main character — atmospheric/environmental shots "
        "are equally important. Anthropomorphic characters are welcome.\n"
        "2. Style: High-end cinematic concept art, surrealist CGI, hyper-realistic 3D render, "
        "or dark fantasy oil painting — pick what best serves the emotional moment.\n"
        "3. Avoid generic tropes. Use emotional symbology instead of literal description.\n\n"
        "### CRITICAL SCENE GROUPING RULE\n"
        "IMPORTANT: Group multiple lyric lines into ONE scene — one scene per VERSE or "
        "thematic paragraph, NOT one scene per sentence. Each scene should represent "
        "8-15 seconds of screen time. Target 10-20 scenes total for a 4-minute song. "
        "Fewer, richer scenes are far better than many thin ones.\n\n"
        "### OUTPUT FORMAT\n"
        "Return ONLY a raw JSON list. No markdown, no ```json wrapper.\n"
        "Required keys per scene:\n"
        "  - `start` (float): timestamp in seconds of the FIRST lyric line in this group\n"
        "  - `text` (string): ALL the lyric lines grouped into this scene\n"
        "  - `scene_description` (string): detailed 200+ word visual prompt for an image "
        "generator — subject, setting, lighting, artistic style, camera angle\n"
        "  - `action_sequence` (string): MAXIMUM 100 WORDS. One short sentence describing "
        "only camera movement and subject motion. Examples: "
        "'Camera slowly pushes in while embers drift upward as Smoke curls upward as the camera holds steady.' "
        "'Figure turns slowly, fabric rippling in the breeze, steady camera shot.' "
        "NO visual descriptions, NO lighting, NO scene content — actions only.\n\n"
        "Describe a 3-12 second SUSTAINED shot, not a quick cut.\n\n"
        "### EXAMPLES\n"
        f"Scene descriptions reference:\n{SCENE_DESCRIPTIONS}\n\n"
        f"Action sequences reference:\n{ACTION_SEQUENCES}\n\n"
        "### TASK\n"
        "Generate the JSON scene list. Remember: group by verse/paragraph, not by sentence. "
        "Each scene = one sustained cinematic shot lasting 3-12 seconds."
    )
    print("Requesting scenes from OpenAI...")
    result = get_openai_prompt_response(
        prompt, config, openai_model=config["openai_model"], temperature=0.85
    )
    if not result:
        print("ERROR: OpenAI returned an empty response.")
        return []
    parsed_scenes = safe_json_parse(result)
    if not parsed_scenes:
        print("ERROR: Failed to parse scenes.")
        print("--- RAW OUTPUT ---")
        print(result)
        print("--- END ---")
        return []
    print(f"Successfully parsed {len(parsed_scenes)} scenes.")
    return parsed_scenes
 
 
def revise_scenes(scenes, character_profile, config: dict):
    simplified = [
        {"scene_description": s.get("scene_description", ""),
         "action_sequence":   s.get("action_sequence", "")}
        for s in scenes
    ]
    prompt = (
        "Enrich every scene_description to 200+ words.\n"
        "Keep every action_sequence SHORT — maximum 2 sentences, under 100 words.\n"
        "The action_sequence is an LTX video motion prompt describing ONLY camera movement\n"
        "and subject motion. It must NOT describe appearance, lighting, or scene content.\n"
        "Good examples:\n"
        "'Camera slowly pushes in while embers drift upward as Smoke curls upward as the camera holds steady.'\n"
        "'Figure turns slowly, fabric rippling in the breeze, steady camera shot.'\n"
        "Return a JSON list with keys: scene_description, action_sequence.\n"
        "Same order as input.\n\n"
        f"Input: {json.dumps(simplified, indent=2)}"
    )
    print("Requesting scene revision from OpenAI...")
    result = get_openai_prompt_response(
        prompt, config, openai_model=config["openai_model"], temperature=0.7
    )
    revised_list = safe_json_parse(result)
    if not revised_list:
        print("WARNING: Revision failed to parse. Keeping original scenes.")
        return scenes
    print(f"Merging {len(revised_list)} revised scenes into {len(scenes)} originals...")
    for i, orig in enumerate(scenes):
        if i < len(revised_list):
            rev = revised_list[i]
            if rev.get("scene_description"):
                orig["scene_description"] = rev["scene_description"]
            if rev.get("action_sequence"):
                orig["action_sequence"] = rev["action_sequence"]
    return scenes
 
 
def enforce_scene_diversity(scenes, character_profile, config: dict):
    print(f"Running Diversity Check on {len(scenes)} scenes...")
    forced_locations = [
        "A neon-lit NOODLE BAR in a cyberpunk slum", "The claustrophobic BURIAL CHAMBER",
        "A floating OBSERVATORY", "A steam locomotive cab", "A Victorian operating theater",
        "A floating continent edge", "A secret alchemist's lab", "A WW2 bomber interior",
        "A gladiator's cell", "A diner at the end of the universe", "A samurai dojo",
        "The Palace of Versailles mirrors", "A bioluminescent coral reef", "A medieval dungeon",
        "A high-speed maglev train", "A smoky billiard hall", "A highway overpass",
        "A mirrored elevator", "A subway car", "An abandoned warehouse", "A rooftop edge",
        "A fire escape", "A pier", "A diner booth", "A tunnel", "A stairwell",
        "A fenced court", "A server room", "A bridge", "A parking garage",
    ]
    random.shuffle(forced_locations)
    for i in range(1, len(scenes)):
        prev_desc  = scenes[i - 1].get("scene_description", "")
        curr_desc  = scenes[i].get("scene_description", "")
        prev_text  = scenes[i - 1].get("text", "")
        curr_text  = scenes[i].get("text", "")
        visual_sim   = similarity_ratio(prev_desc, curr_desc)
        lyrics_match = curr_text.strip() == prev_text.strip() and curr_text.strip() != ""
        if lyrics_match or visual_sim > 0.70:
            print(f" > Repetition at Scene {i}. Rewriting...")
            is_b_roll = random.random() < 0.40
            loc = forced_locations[i % len(forced_locations)]
            if is_b_roll:
                variation_prompt = (
                    f"Revise to an ATMOSPHERIC B-ROLL SHOT at: {loc}\n"
                    "No main character. Cinematic, detailed.\n"
                    f"Original: {curr_desc}\n"
                    'Return JSON with keys: "scene_description", "action_sequence".'
                )
            else:
                variation_prompt = (
                    f'Include the MAIN CHARACTER ("{character_profile}") at: {loc}\n'
                    "Action: active (Running, Dancing).\n"
                    f"Original: {curr_desc}\n"
                    'Return JSON with keys: "scene_description", "action_sequence".'
                )
            try:
                res = get_openai_prompt_response(
                    variation_prompt, config,
                    openai_model=config["openai_model"], temperature=0.95,
                )
                new_data = safe_json_parse(res)
                if isinstance(new_data, list) and new_data:
                    new_data = new_data[0]
                if isinstance(new_data, dict):
                    if "scene_description" in new_data:
                        scenes[i]["scene_description"] = new_data["scene_description"]
                    if "action_sequence" in new_data:
                        scenes[i]["action_sequence"] = new_data["action_sequence"]
            except Exception as e:
                print(f"   > Failed to fix scene {i}: {e}")
    return scenes
 
 
# =============================================================================
# Scene pipeline: transcribe -> merge -> LLM scenes -> split
# =============================================================================
def process_audio_scenes(audio_file: str, config: dict):
    max_duration_seconds = config["ltx_max_scene_seconds"]
    audio_basename = os.path.splitext(os.path.basename(audio_file))[0]
    timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
    unique_id  = f"{audio_basename}_{timestamp}"
 
    audio_images_dir = os.path.join(config["base_working_dir"], unique_id)
    audio_videos_dir = os.path.join(config["base_video_dir"],   unique_id)
    os.makedirs(audio_images_dir, exist_ok=True)
    os.makedirs(audio_videos_dir, exist_ok=True)
 
    # --- STEP 1: load audio (ffmpeg + soundfile, no librosa/numba) -----------
    print("Transcribing audio using Whisper (Sliding Window for Full Song)...")
    processor, whisper_model = get_audio_pipe()
    full_audio, sampling_rate = load_audio_mono_16k(audio_file)
    total_duration = len(full_audio) / sampling_rate
    print(f"Total audio duration: {total_duration:.2f}s")
 
    # --- STEP 2: sliding-window Whisper transcription ------------------------
    segments     = []
    chunk_length = 30
    overlap      = 5
    step         = chunk_length - overlap
 
    for start_sec in range(0, int(total_duration), step):
        end_sec = min(start_sec + chunk_length, total_duration)
        print(f" > Processing chunk: {start_sec}s to {end_sec}s")
        chunk_audio = full_audio[
            int(start_sec * sampling_rate): int(end_sec * sampling_rate)
        ]
        inputs = processor(
            chunk_audio, sampling_rate=16000,
            return_tensors="pt", return_attention_mask=True,
        )
        input_features = inputs.input_features.to("cuda").to(dtype)
        attention_mask = inputs.attention_mask.to("cuda")
        with torch.no_grad():
            predicted_ids = whisper_model.generate(
                input_features,
                attention_mask=attention_mask,
                return_timestamps=True,
                task="transcribe",
                max_new_tokens=444,              # 448 total - 4 special tokens
                no_speech_threshold=0.6,
                logprob_threshold=-1.0,
                compression_ratio_threshold=2.4,
                condition_on_prev_tokens=False,  # prevents hallucination chaining
                temperature=0.0,
            )
        # Use processor.decode (singular) on predicted_ids[0] to avoid the
        # "Boolean value of Tensor with more than one value is ambiguous" error
        # that batch_decode raises when passing a 2D tensor to _decode_with_timestamps.
        chunk_text = processor.decode(
            predicted_ids[0],
            skip_special_tokens=False,
            decode_with_timestamps=True,
        )
        raw_chunks = re.findall(
            r"<\|(\d+\.\d+)\|>(.*?)(?=<\|\d+\.\d+\|>|$)", chunk_text
        )
        for ts_str, txt in raw_chunks:
            clean_txt   = txt.strip()
            relative_ts = float(ts_str)
            if start_sec + chunk_length < total_duration and relative_ts >= step:
                continue
            # Filter hallucinated repeating phrases
            words = clean_txt.split()
            if len(words) > 6:
                trigram = " ".join(words[:3]).lower()
                if clean_txt.lower().count(trigram) > 2:
                    print(f"   > Skipping hallucinated segment at {relative_ts + start_sec:.1f}s")
                    continue
            if clean_txt:
                segments.append((relative_ts + start_sec, clean_txt))
 
    del processor, whisper_model, full_audio
    reset_memory(device)
 
    # --- STEP 3: merge short segments into verse-level chunks ----------------
    segments.sort(key=lambda x: x[0])
    raw_segments = segments.copy()   # keep unmerged copy for retry fallback
 
    MIN_SEGMENT_SECONDS = 3.0
    merged_segments = []
    if segments:
        current_text  = []
        current_start = segments[0][0]
        for idx, (ts, txt) in enumerate(segments):
            current_text.append(txt)
            is_last  = (idx == len(segments) - 1)
            next_ts  = segments[idx + 1][0] if not is_last else total_duration
            span     = next_ts - current_start
            if span >= MIN_SEGMENT_SECONDS or is_last:
                merged_segments.append((current_start, " ".join(current_text)))
                current_start = next_ts
                current_text  = []
        segments = merged_segments
 
    # --- STEP 4: build LLM input text ----------------------------------------
    text_for_llm   = "".join(f"Start: {s:.2f}, Text: {t}\n" for s, t in segments)
    last_end_value = float(total_duration)
    scenes_file_path = os.path.join(audio_images_dir, "scenes.json")
 
    # --- STEP 5: LLM scene generation ----------------------------------------
    char_profile = generate_character_profile(text_for_llm, config)
    print("Requesting initial scenes from LLM...")
    scenes = None
    try:
        scenes = create_scenes(text_for_llm, char_profile, config)
        # Retry with raw unmerged segments if too few scenes returned
        if not scenes or len(scenes) < 5:
            print(f"WARNING: Only {len(scenes) if scenes else 0} scenes returned. "
                  "Retrying with unmerged segments...")
            raw_text = "".join(f"Start: {s:.2f}, Text: {t}\n" for s, t in raw_segments)
            scenes   = create_scenes(raw_text, char_profile, config)
    except Exception as e:
        print(f"LLM Attempt 1 failed: {e}. Retrying...")
        try:
            scenes = create_scenes(text_for_llm, char_profile, config)
        except Exception as e:
            print(f"LLM Critical Failure: {e}")
            return "", audio_images_dir, audio_videos_dir, last_end_value, timestamp
 
    if not scenes:
        return "", audio_images_dir, audio_videos_dir, last_end_value, timestamp
 
    # --- STEP 6: split scenes longer than max_duration_seconds ---------------
    print(f"Splitting scenes longer than {max_duration_seconds}s...")
    new_scenes = []
    for i, scene in enumerate(scenes):
        start_time = float(scene.get("start", 0.0))
        if i < len(scenes) - 1:
            end_time = float(scenes[i + 1].get("start", start_time + max_duration_seconds))
        else:
            end_time = last_end_value
        duration = end_time - start_time
        while duration > max_duration_seconds:
            split_scene          = scene.copy()
            split_scene["start"] = start_time
            new_scenes.append(split_scene)
            start_time += max_duration_seconds
            duration    = end_time - start_time
        if duration > 0:
            split_scene          = scene.copy()
            split_scene["start"] = start_time
            new_scenes.append(split_scene)
    scenes = new_scenes
 
    # --- STEP 7: revise and diversify ----------------------------------------
    if scenes:
        try:
            print("Revising scene descriptions and enforcing diversity...")
            scenes = revise_scenes(scenes, char_profile, config)
            scenes = enforce_scene_diversity(scenes, char_profile, config)
        except Exception as e:
            print(f"Warning: Scene refinement skipped due to error: {e}")
 
    with open(scenes_file_path, "w") as f:
        json.dump(scenes, f, indent=2)
    print(f"Successfully processed {len(scenes)} scenes.")
    return scenes, audio_images_dir, audio_videos_dir, last_end_value, timestamp
 
 
# =============================================================================
# Z-Image still generation per scene
# =============================================================================
def process_audio_images(config: dict, scenes, audio_images_dir):
    print("Loading Z-Image pipeline and generating stills...")
    pipe = load_zimage_models()
    resolution      = (HEIGHT, WIDTH)
    negative_prompt = (
        "worst quality, low quality, worst aesthetic, old, blurry, lowres, signature, "
        "artist name, watermark, username, sketch, logo, furry, text, speech bubble, "
        "symbols, words, letteres"
    )
    for image_num, scene in enumerate(scenes, 1):
        description  = scene.get("scene_description") or "an animated scene in a music video"
        image_prompt = THEME + ". " + description
        print(f"Generating Image {image_num}: {image_prompt[:60]}...")
        image, _ = generate_image(pipe, image_prompt, negative_prompt, resolution, -1)
        image.save(
            os.path.join(audio_images_dir, f"image_{str(image_num).zfill(2)}.jpg"),
            dpi=(300, 300),
        )
    del pipe
    reset_memory(device)
    print(f"VRAM after Z-Image release: "
          f"{torch.cuda.memory_allocated() / 1e9:.1f} GB alloc, "
          f"{torch.cuda.memory_reserved() / 1e9:.1f} GB reserved")
 
 
# =============================================================================
# LTX-2.3 asset discovery
# =============================================================================
def find_ltx2_assets(config: dict) -> dict:
    """
    Locate the LTX-2 model files without importing anything heavy.
    Search order: explicit config paths → models/ subdirs → repo root → HF cache.
    Newest file wins when multiple matches exist.
    """
    repo   = os.path.expanduser(LTX_ROOT)
    result = {
        "checkpoint": "", "checkpoint_is_distilled": False,
        "spatial_upsampler": "", "distilled_lora": "", "gemma_root": "",
        "ready": False, "missing": [],
    }
 
    roots = []
    repo_models = os.path.join(repo, "models")
    if os.path.isdir(repo_models):
        roots.append(repo_models)
    if os.path.isdir(repo):
        roots.append(repo)
    hf_cache = os.path.expanduser(
        os.environ.get("HF_HOME",
            os.environ.get("HUGGINGFACE_HUB_CACHE", "~/.cache/huggingface"))
    )
    if os.path.isdir(hf_cache):
        roots.append(hf_cache)
 
    def _find(explicit: str, *patterns, exclude=()):
        if explicit:
            p = os.path.expanduser(explicit)
            if os.path.exists(p):
                return os.path.abspath(p)
        hits = []
        import fnmatch
        for root in roots:
            for dirpath, _, filenames in os.walk(root):
                for fn in filenames:
                    for pat in patterns:
                        if fnmatch.fnmatch(fn.lower(), pat.lower()):
                            if not any(x in fn.lower() for x in exclude):
                                hits.append(os.path.join(dirpath, fn))
        if not hits:
            return ""
        return max(hits, key=os.path.getmtime)
 
    # Checkpoint: prefer distilled, then dev. On no native FP8, prefer bf16 over fp8.
    explicit_ckpt = config.get("ltx_checkpoint", "")
    result["checkpoint"] = _find(
        explicit_ckpt, "ltx-2*distilled*.safetensors",
        exclude=("lora", "upscaler", "upsampler"),
    )
    if not result["checkpoint"]:
        result["checkpoint"] = _find(
            explicit_ckpt, "ltx-2*dev*.safetensors",
            exclude=("lora", "upscaler", "upsampler"),
        )
    if result["checkpoint"]:
        result["checkpoint_is_distilled"] = (
            "distilled" in os.path.basename(result["checkpoint"]).lower()
        )
    else:
        result["missing"].append("No LTX-2 checkpoint found")
 
    result["spatial_upsampler"] = _find(
        config.get("ltx_spatial_upsampler", ""),
        "*spatial-upscaler-x2*.safetensors", "*spatial-upscaler*.safetensors",
    )
    if not result["spatial_upsampler"]:
        result["missing"].append("Spatial upscaler not found")
 
    result["distilled_lora"] = _find(
        config.get("ltx_distilled_lora", ""),
        "*distilled-lora*.safetensors",
    )
    if not result["distilled_lora"] and not result["checkpoint_is_distilled"]:
        result["missing"].append("Distilled LoRA not found (required with dev checkpoint)")
 
    gemma_explicit = config.get("ltx_gemma_root", "")
    if gemma_explicit and os.path.isdir(os.path.expanduser(gemma_explicit)):
        result["gemma_root"] = os.path.abspath(os.path.expanduser(gemma_explicit))
    else:
        for root in roots:
            for dirpath, _, filenames in os.walk(root):
                if "gemma" in dirpath.lower() and "config.json" in filenames:
                    result["gemma_root"] = dirpath
                    break
            if result["gemma_root"]:
                break
    if not result["gemma_root"]:
        result["missing"].append("Gemma text-encoder directory not found")
 
    result["ready"] = not result["missing"]
    return result
 
 
# =============================================================================
# LTX-2.3 pipeline helpers
# =============================================================================
def call_supported(fn, /, **kwargs):
    """Call fn with only the kwargs its signature accepts (drift tolerance)."""
    sig = inspect.signature(fn)
    if any(p.kind == p.VAR_KEYWORD for p in sig.parameters.values()):
        return fn(**kwargs)
    return fn(**{k: v for k, v in kwargs.items() if k in sig.parameters})
 
 
def _build_quantization_policy(quant_str: str, ckpt_path: str):
    ckpt_name = os.path.basename(ckpt_path).lower()
    if "fp8" in ckpt_name:
        print("[LTX2] Pre-quantized FP8 checkpoint — quantization=None.")
        return None

    if not quant_str or quant_str.lower() in ("none", ""):
        return None

    quant_str = quant_str.lower()

    if quant_str == "fp8-cast":
        # Cache the policy so build_policy() only mmaps the checkpoint once
        # per process, not once per pipeline instantiation.
        cache_attr = "_ltx2_fp8_cast_policy_cache"
        cached = globals().get(cache_attr)
        if cached is not None:
            print("[LTX2] fp8-cast policy: using cached policy (no mmap).")
            return cached

        try:
            from ltx_core.quantization.fp8_cast import build_policy
            print("[LTX2] fp8-cast: calling build_policy() — mmaps checkpoint once, then caches.")
            policy = build_policy(ckpt_path)
            globals()[cache_attr] = policy
            print("[LTX2] fp8-cast policy built and cached.")
            return policy
        except Exception as e:
            print(f"[LTX2] build_policy() failed: {e} — running bf16.")
            return None

    print(f"[LTX2] Unknown quantization '{quant_str}' — running bf16.")
    return None
 
 
def _build_offload_mode(offload_str: str):
    """Import OffloadMode with fallback."""
    try:
        from ltx_pipelines.utils.types import OffloadMode
        try:
            return OffloadMode(offload_str)
        except ValueError:
            return OffloadMode[offload_str.upper()]
    except Exception:
        return None
 
 
def _make_image_conditioning(path: str):
    """
    Build the image conditioning input in whatever format this revision accepts.
    Prefers ImageConditioningInput NamedTuple; falls back to plain tuple.
    """
    try:
        from ltx_pipelines.utils.args import ImageConditioningInput
        try:
            return ImageConditioningInput(path, 0, 1.0, 33)  # path, frame, strength, crf
        except TypeError:
            return ImageConditioningInput(path, 0, 1.0)
    except ImportError:
        return (path, 0, 1.0)
 
 
def build_ltx_pipeline(assets: dict, config: dict):
    from ltx_pipelines.a2vid_two_stage import A2VidPipelineTwoStage

    ckpt         = assets["checkpoint"]
    upsampler    = assets["spatial_upsampler"]
    gemma        = assets["gemma_root"]
    is_distilled = assets["checkpoint_is_distilled"]
    ckpt_is_fp8  = "fp8" in os.path.basename(ckpt).lower()

    quant = _build_quantization_policy(config.get("ltx_quantization", "fp8-cast"), ckpt)

    # Pre-quantized fp8 checkpoint CANNOT use block streaming (cpu/disk offload).
    # derive_layout() expects input_scale for every param; BF16 gate layers don't
    # have it, causing KeyError: 'attn1.to_gate_logits.input_scale'.
    # Solution: offload_mode=None (full in-VRAM load) for fp8 checkpoints.
    # On a 4090 the fp8 file is ~29.5 GB — it won't fully fit in 24 GB VRAM alone,
    # but the pipeline loads components sequentially so peak VRAM stays manageable.
    if ckpt_is_fp8:
        print("[LTX2] FP8 checkpoint detected — forcing offload_mode=None "
              "(block streaming is incompatible with pre-quantized fp8 checkpoints).")
        offload = _build_offload_mode("none")
    else:
        offload = _build_offload_mode(config.get("ltx_offload", "cpu"))

    # Load distilled LoRA if specified — for 10Eros use the condsafe variant.
    # Unlike standard distilled checkpoint, 10Eros benefits from the LoRA
    # even though it's a distilled-style merge.
    lora_list = []
    if assets.get("distilled_lora"):
        try:
            from ltx_core.loader import LTXV_LORA_COMFY_RENAMING_MAP, LoraPathStrengthAndSDOps
            lora_list = [LoraPathStrengthAndSDOps(
                assets["distilled_lora"],
                float(config.get("ltx_distilled_lora_strength", 1.0)),
                LTXV_LORA_COMFY_RENAMING_MAP,
            )]
            print(f"[LTX2] LoRA loaded: {os.path.basename(assets['distilled_lora'])} "
                  f"@ strength {config.get('ltx_distilled_lora_strength', 1.0)}")
        except Exception as e:
            print(f"[LTX2] WARNING: Could not build LoRA descriptor: {e}")

    pipeline = call_supported(
        A2VidPipelineTwoStage,
        checkpoint_path        = ckpt,
        distilled_lora         = lora_list,
        spatial_upsampler_path = upsampler,
        gemma_root             = gemma,
        loras                  = [],
        quantization           = quant,
        offload_mode           = offload,
    )
    # Reduce VAE peak VRAM — especially helpful with offload_mode=None
    try:
        if hasattr(pipeline, 'video_decoder') and hasattr(pipeline.video_decoder, '_vae'):
            pipeline.video_decoder._vae.enable_slicing()
            pipeline.video_decoder._vae.enable_tiling()
            print("[LTX2] VAE slicing + tiling enabled")
    except Exception as e:
        print(f"[LTX2] VAE slicing not available: {e}")
    return pipeline, is_distilled
 
 
def nearest_8k_plus_1(target_frames: float, min_frames: int, max_frames: int) -> int:
    """LTX requires frame counts of the form 8*k + 1."""
    k = round((target_frames - 1) / 8)
    k = max(k, (min_frames - 1) // 8)
    k = min(k, (max_frames - 1) // 8)
    k = max(k, 1)
    return 8 * k + 1
 
 
def generate_scene_clip(pipeline, is_distilled, assets, config,
                        image_path, audio_slice_path, prompt,
                        num_frames, seed, out_path):
    from ltx_core.components.guiders import MultiModalGuiderParams
    from ltx_core.model.video_vae import TilingConfig, get_video_chunks_number
    from ltx_pipelines.utils.media_io import encode_video

    fps          = float(config["ltx_fps"])
    tiling       = TilingConfig.default()
    video_chunks = get_video_chunks_number(num_frames, tiling)
    images       = [_make_image_conditioning(image_path)]

    if is_distilled:
        try:
            from ltx_pipelines.utils.constants import DISTILLED_SIGMAS
            s1_sigmas = DISTILLED_SIGMAS
        except ImportError:
            s1_sigmas = None
        gp = MultiModalGuiderParams(
            cfg_scale=1.0, stg_scale=0.0, rescale_scale=0.0,
            modality_scale=1.0, skip_step=0, stg_blocks=[],
        )
    else:
        s1_sigmas = None
        gp = MultiModalGuiderParams(
            cfg_scale=3.0, stg_scale=1.0, rescale_scale=0.7,
            modality_scale=float(config.get("ltx_a2v_guidance_scale", 3.0)),
            skip_step=0, stg_blocks=[29],
        )

    # Wrap EVERYTHING — pipeline call AND encode_video — in a single
    # no_grad block. encode_video runs VAE decode + spatial upsampler;
    # without no_grad it builds an autograd graph that pins the full
    # video tensor in VRAM until Python GC runs, causing OOM on scene 2+.
    with torch.no_grad():
        video, audio = call_supported(
            pipeline.__call__,
            prompt              = _truncate_prompt(prompt, max_words=300),
            negative_prompt     = _truncate_prompt(config["ltx_negative_prompt"], max_words=150),
            seed                = seed,
            height              = config["ltx_height"],
            width               = config["ltx_width"],
            num_frames          = num_frames,
            frame_rate          = fps,
            num_inference_steps = int(config.get("ltx_num_inference_steps", 8)),
            video_guider_params = gp,
            images              = images,
            tiling_config       = tiling,
            enhance_prompt      = bool(config.get("ltx_enhance_prompt", False)),
            audio_path          = audio_slice_path,
            audio_start_time    = 0.0,
            audio_max_duration  = num_frames / fps,
            stage_1_sigmas      = s1_sigmas,
        )

        # encode_video INSIDE no_grad — prevents autograd graph retention
        encode_video(
            video               = video,
            fps                 = fps,
            audio               = audio,
            output_path         = out_path,
            video_chunks_number = video_chunks,
        )

    # Move tensors to CPU before deleting so VRAM is freed immediately
    if isinstance(video, torch.Tensor):
        video = video.cpu()
    if isinstance(audio, torch.Tensor):
        audio = audio.cpu()
    del video, audio
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()        

    # Tensors are now detached (no_grad context) so del + empty_cache
    # actually frees VRAM immediately rather than waiting for GC.
    print(f"  VRAM after cleanup: "
          f"{torch.cuda.memory_allocated()/1e9:.1f} GB alloc, "
          f"{torch.cuda.memory_reserved()/1e9:.1f} GB reserved")

    return out_path
 
 
# =============================================================================
# Main video generation loop
# =============================================================================
def process_audio_video_ltx(config: dict, scenes, audio_file: str,
                             audio_images_dir: str, audio_videos_dir: str,
                             last_end_value: float, timestamp: str,
                             start_image: int = 0):

    # ---- Diagnostic: verify scene/image/audio counts up front --------------
    print(f"\n{'='*60}")
    print(f"VIDEO PHASE START")
    print(f"  Scenes in list:  {len(scenes)}")
    print(f"  Audio file:      {audio_file}")
    print(f"  last_end_value:  {last_end_value:.2f}s")
    print(f"  start_image:     {start_image}")

    # Count how many image files actually exist
    existing_images = sorted([
        f for f in os.listdir(audio_images_dir)
        if f.startswith("image_") and f.endswith(".jpg")
    ])
    print(f"  Image files found: {len(existing_images)}")
    if len(existing_images) != len(scenes):
        print(f"  WARNING: image count ({len(existing_images)}) != "
              f"scene count ({len(scenes)}) — mismatch will cause skips!")
    print(f"{'='*60}\n")

    # ---- PASS 1: Write ALL audio slices first, independently ----------------
    # This makes it easy to verify A/V alignment before spending GPU time.
    audio_slices_dir = os.path.join(audio_videos_dir, "audio_slices")
    os.makedirs(audio_slices_dir, exist_ok=True)

    fps              = float(config["ltx_fps"])
    min_frames       = nearest_8k_plus_1(config["ltx_min_scene_seconds"] * fps, 9, 10_000)
    max_frames       = nearest_8k_plus_1(config["ltx_max_scene_seconds"] * fps, 9, 10_000)
    default_duration = 12.0

    # Pre-compute timing for every scene and cache it so the render loop
    # uses identical values (no re-computation drift).
    scene_timings = []   # list of (current_start, clip_duration, num_frames)
    for i, scene in enumerate(scenes):
        current_start = scene.get("start")
        if current_start is None:
            current_start = (
                0.0 if i == 0
                else scene_timings[i - 1][0] + scene_timings[i - 1][1]
            )
            scene["start"] = current_start

        if i + 1 < len(scenes):
            next_start_time = scenes[i + 1].get("start")
            if next_start_time is None:
                next_start_time        = current_start + default_duration
                scenes[i + 1]["start"] = next_start_time
        else:
            next_start_time = last_end_value

        scene_duration = next_start_time - current_start
        if scene_duration <= 0:
            print(f"Warning: Scene {i+1} has invalid duration "
                  f"({scene_duration:.2f}s). Forcing {default_duration}s.")
            scene_duration = default_duration

        num_frames    = nearest_8k_plus_1(scene_duration * fps, min_frames, max_frames)
        clip_duration = num_frames / fps
        scene_timings.append((current_start, clip_duration, num_frames))

    print("Writing audio slices...")
    slice_paths = []
    for i, (current_start, clip_duration, num_frames) in enumerate(scene_timings):
        slice_path = os.path.join(
            audio_slices_dir, f"scene_{str(i + 1).zfill(2)}.wav"
        )
        try:
            slice_audio(audio_file, current_start, clip_duration, slice_path)
            slice_paths.append(slice_path)
            print(f"  Scene {i+1:02d}: start={current_start:.2f}s  "
                  f"dur={clip_duration:.2f}s  frames={num_frames}  -> {os.path.basename(slice_path)}")
        except Exception as e:
            print(f"  Scene {i+1:02d}: FAILED to write audio slice: {e}")
            slice_paths.append(None)

    print(f"\nAudio slices written: "
          f"{sum(1 for p in slice_paths if p)} / {len(scenes)}\n")

    # ---- PASS 2: Discover assets and load pipeline --------------------------
    print("Discovering LTX-2.3 model assets...")
    assets = find_ltx2_assets(config)
    # 10Eros is a distilled-style merge — treat it as distilled
    if "10eros" in os.path.basename(assets["checkpoint"]).lower() or \
       "10eros" in config.get("ltx_checkpoint", "").lower():
        assets["checkpoint_is_distilled"] = True
        print("[LTX2] 10Eros detected — treating as distilled checkpoint.")

    if not assets["ready"]:
        missing = "\n  - ".join(assets["missing"])
        raise RuntimeError(
            f"LTX-2.3 assets not ready:\n  - {missing}\n\n"
            f"Model files should be in {LTX_MODELS}/ltx-2.3/ and "
            f"{LTX_MODELS}/gemma-3-12b/\n"
            "Required files:\n"
            "  ltx-2.3-22b-distilled-1.1.safetensors  (or the fp8 variant)\n"
            "  ltx-2.3-spatial-upscaler-x2-1.1.safetensors\n"
            "  ltx-2.3-22b-distilled-lora-384-1.1.safetensors\n"
            "  gemma-3-12b/  (google/gemma-3-12b-it-qat-q4_0-unquantized)\n"
            "Install packages once:\n"
            f"  pip install -e {LTX_ROOT}/packages/ltx-core\n"
            f"  pip install -e {LTX_ROOT}/packages/ltx-pipelines"
        )

    print(f"[LTX2] Checkpoint:    {os.path.basename(assets['checkpoint'])}")
    print(f"[LTX2] Upsampler:     {os.path.basename(assets['spatial_upsampler'])}")
    print(f"[LTX2] Gemma:         {os.path.basename(assets['gemma_root'])}")
    print(f"[LTX2] DistilledCkpt: {assets['checkpoint_is_distilled']}")
    if assets.get("distilled_lora"):
        print(f"[LTX2] LoRA:          {os.path.basename(assets['distilled_lora'])}")

    print("Loading LTX-2.3 A2Vid pipeline (done once, reused for all scenes)...")
    pipeline, is_distilled = build_ltx_pipeline(assets, config)

    # ---- PASS 3: Render each scene ------------------------------------------
    print(f"\nRendering {len(scenes)} scenes x {config['ltx_num_variations']} variations...\n")
    for i, scene in enumerate(scenes):
        if i < start_image - 1:
            print(f"Scene {i+1:02d}: skipping (before start_image={start_image})")
            continue

        audio_slice_path = slice_paths[i]
        if audio_slice_path is None or not os.path.exists(audio_slice_path):
            print(f"Scene {i+1:02d}: SKIPPING — audio slice missing")
            continue

        image_input = os.path.join(
            audio_images_dir, f"image_{str(i + 1).zfill(2)}.jpg"
        )
        if not os.path.exists(image_input):
            print(f"Scene {i+1:02d}: SKIPPING — still image missing: {image_input}")
            continue

        current_start, clip_duration, num_frames = scene_timings[i]
        prompt = scene.get("action_sequence") or "an animated image"

        for v in range(config["ltx_num_variations"]):
            seed     = secrets.randbits(31)
            out_name = (f"v_ltx_{str(i + 1).zfill(2)}_var{v + 1}_{timestamp}.mp4")
            out_path = os.path.join(audio_videos_dir, out_name)
            print(
                f"Scene {i+1:02d} var {v+1}: "
                f"start={current_start:.2f}s  dur={clip_duration:.2f}s  "
                f"frames={num_frames}  seed={seed}"
            )
            try:
                generate_scene_clip(
                    pipeline, is_distilled, assets, config,
                    image_input, audio_slice_path,
                    prompt, num_frames, seed, out_path,
                )
                print(f"  -> saved: {os.path.basename(out_path)}")
            except torch.cuda.OutOfMemoryError:
                # OOM: flush everything and skip this scene rather than
                # crashing the whole pipeline.
                print(f"  -> OOM on scene {i+1} var {v+1} — flushing and continuing.")
                gc.collect()
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            except Exception:
                traceback.print_exc()
                print(f"  -> FAILED: scene {i+1} var {v+1}")
                gc.collect()
                torch.cuda.empty_cache()
            time.sleep(1)

    del pipeline
    reset_memory(device)
    print(f"\nVideo phase complete. Output: {audio_videos_dir}")
 
 
# =============================================================================
# Orchestration
# =============================================================================
def process_all_audios(audio_file, config: dict):
    print(f"Processing audio file: {audio_file}")
    scenes, audio_images_dir, audio_videos_dir, last_end_value, timestamp = \
        process_audio_scenes(audio_file, config)
    print(f"{len(scenes)} scenes:\n{json.dumps(scenes, indent=4)}")
    print(f"last_end_value: {last_end_value}  timestamp: {timestamp}")
    process_audio_images(config, scenes, audio_images_dir)
    return config, scenes, audio_images_dir, audio_videos_dir, last_end_value, timestamp
 
 
def create_video(config, audio_file):
    config, scenes, audio_images_dir, audio_videos_dir, last_end_value, timestamp = \
        process_all_audios(audio_file, config)
    process_audio_video_ltx(
        config, scenes, audio_file,
        audio_images_dir, audio_videos_dir,
        last_end_value, timestamp,
        start_image=0,
    )

In [4]:
# run new systems
for audio_file in CONFIG["audio_files"]:
    create_video(CONFIG, audio_file)
    reset_memory(device)


Processing audio file: /mnt/d/Share/Audio/BombsBursting.mp3
Transcribing audio using Whisper (Sliding Window for Full Song)...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Total audio duration: 283.52s
 > Processing chunk: 0s to 30s


[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The cus

 > Processing chunk: 25s to 55s


...

  VRAM after cleanup: 0.0 GB alloc, 0.0 GB reserved
  -> saved: v_ltx_01_var1_20260711_123139.mp4
Scene 02 var 1: start=12.00s  dur=3.04s  frames=73  seed=725025141


 50%|██████████████████████████████████████████████████████████████████████████▌                                                                          | 1/2 [00:00<00:00,  2.65it/s]


  VRAM after cleanup: 0.0 GB alloc, 0.0 GB reserved
  -> saved: v_ltx_02_var1_20260711_123139.mp4
Scene 03 var 1: start=14.60s  dur=4.71s  frames=113  seed=1709395055


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:08<00:00,  3.00s/it]


KeyboardInterrupt: 

## Run Previous Generation

In [8]:
# run saved ProphetFist_20251211_192956
timestamp = '20260711_123139'
title = 'BombsBursting'
start_image = 0
scenes_file_path = f'./images/{title}_{timestamp}/scenes.json'
audio_images_dir = f'./images/{title}_{timestamp}'
audio_videos_dir = f'./output/{title}_{timestamp}'

last_end_value = 283.52 

with open(scenes_file_path, "r") as scenes_file:
    scenes = json.load(scenes_file)

process_audio_video_ltx(CONFIG, scenes, CONFIG["audio_files"][0], audio_images_dir, audio_videos_dir, last_end_value, timestamp, start_image)


VIDEO PHASE START
  Scenes in list:  35
  Audio file:      /mnt/d/Share/Audio/BombsBursting.mp3
  last_end_value:  283.52s
  start_image:     0
  Image files found: 35

Writing audio slices...
  Scene 01: start=0.00s  dur=12.04s  frames=289  -> scene_01.wav
  Scene 02: start=12.00s  dur=3.04s  frames=73  -> scene_02.wav
  Scene 03: start=14.60s  dur=4.71s  frames=113  -> scene_03.wav
  Scene 04: start=19.42s  dur=4.71s  frames=113  -> scene_04.wav
  Scene 05: start=24.04s  dur=5.38s  frames=129  -> scene_05.wav
  Scene 06: start=29.36s  dur=7.04s  frames=169  -> scene_06.wav
  Scene 07: start=36.24s  dur=12.04s  frames=289  -> scene_07.wav
  Scene 08: start=48.24s  dur=5.71s  frames=137  -> scene_08.wav
  Scene 09: start=53.90s  dur=3.38s  frames=81  -> scene_09.wav
  Scene 10: start=57.38s  dur=5.04s  frames=121  -> scene_10.wav
  Scene 11: start=62.56s  dur=12.04s  frames=289  -> scene_11.wav
  Scene 12: start=74.56s  dur=3.04s  frames=73  -> scene_12.wav
  Scene 13: start=75.00s  d

 83%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏                        | 5/6 [00:04<00:00,  1.05it/s]


  VRAM after cleanup: 0.3 GB alloc, 0.4 GB reserved
  -> saved: v_ltx_01_var1_20260711_123139.mp4
Scene 02 var 1: start=12.00s  dur=3.04s  frames=73  seed=1503796073


 50%|██████████████████████████████████████████████████████████████████████████▌                                                                          | 1/2 [00:00<00:00,  2.81it/s]


  VRAM after cleanup: 0.3 GB alloc, 0.4 GB reserved
  -> saved: v_ltx_02_var1_20260711_123139.mp4
Scene 03 var 1: start=14.60s  dur=4.71s  frames=113  seed=359097418


...

  VRAM after cleanup: 0.3 GB alloc, 0.4 GB reserved
  -> saved: v_ltx_33_var1_20260711_123139.mp4
Scene 34 var 1: start=266.42s  dur=8.71s  frames=209  seed=132640706


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.37it/s]


  VRAM after cleanup: 0.3 GB alloc, 0.4 GB reserved
  -> saved: v_ltx_34_var1_20260711_123139.mp4
Scene 35 var 1: start=275.00s  dur=8.38s  frames=201  seed=2090968489


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.41it/s]


  VRAM after cleanup: 0.3 GB alloc, 0.4 GB reserved
  -> saved: v_ltx_35_var1_20260711_123139.mp4

Video phase complete. Output: ./output/BombsBursting_20260711_123139
